In [ ]:
# !pip install langchain==0.3.27 langchain-openai==0.3.33 langchain-community==0.3.24

In [1]:
import warnings
warnings.filterwarnings('ignore')

In [2]:
import langchain, langchain_community
print(langchain.__version__)
print(langchain_community.__version__)

0.1.13
0.0.29


In [3]:
import os
import json
from langchain_openai import ChatOpenAI
from langchain.schema import HumanMessage, SystemMessage
from textwrap import indent
import openai

import utils

In [4]:
llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)

In [5]:
# Define reasoning templates for different problem types
REASONING_TEMPLATES = {
    "math_word_problem": """
Let's think through this step by step:
1. First, identify what the problem is asking for
2. Extract the relevant numbers and variables
3. Determine the mathematical operations needed
4. Set up the equation or calculation
5. Solve step by step
6. Verify the answer makes sense
""",

    "logical_deduction": """
Let's reason through this logically:
1. List all the given facts and constraints
2. Identify relationships between elements
3. Look for contradictions or implications
4. Make inferences from the available information
5. Build conclusions step by step
6. Check for consistency
""",

    "code_debugging": """
Let's debug this systematically:
1. Understand what the code should do vs what it actually does
2. Identify the specific error or unexpected behavior
3. Trace through the code execution step by step
4. Check variable values and data flow
5. Isolate the problematic section
6. Propose and test fixes
"""
}

In [6]:
# Example problems - you can replace these dynamically
problems = {
    "math_problem": "If a store has 15 apples and sells 8, then receives a shipment of 12 more apples, how many apples do they have now?",
    
    "logic_puzzle": "There are three people: Alice, Bob, and Charlie. Alice is taller than Bob. Charlie is shorter than Bob. Who is the tallest?",
    
    "debugging_issue": "This Python function should return the sum of even numbers in a list, but it's returning incorrect results:\n\n```python\ndef sum_even_numbers(numbers):\n    total = 0\n    for num in numbers:\n        if num % 2 == 0:\n            total += num\n    return total\n\n# Test case: sum_even_numbers([1, 2, 3, 4, 5, 6]) returns 9 instead of 12\n```"
}


In [7]:
# Select a problem to solve
selected_problem = problems["math_problem"]
problem_type = "math_word_problem"

In [8]:
# Build the chain of thought prompt
prompt = f"""
You are an AI reasoning assistant. Your task is to solve problems using clear, step-by-step chain of thought reasoning.

[PROBLEM]
{selected_problem}

[REASONING APPROACH]
{REASONING_TEMPLATES[problem_type]}

Please work through the problem systematically, showing your reasoning at each step before providing the final answer.

Format your response as:
Step 1: [Your first reasoning step]
Step 2: [Your next reasoning step]
...
Final Answer: [Your conclusive answer]
"""

In [9]:
print(prompt)


You are an AI reasoning assistant. Your task is to solve problems using clear, step-by-step chain of thought reasoning.

[PROBLEM]
If a store has 15 apples and sells 8, then receives a shipment of 12 more apples, how many apples do they have now?

[REASONING APPROACH]

Let's think through this step by step:
1. First, identify what the problem is asking for
2. Extract the relevant numbers and variables
3. Determine the mathematical operations needed
4. Set up the equation or calculation
5. Solve step by step
6. Verify the answer makes sense


Please work through the problem systematically, showing your reasoning at each step before providing the final answer.

Format your response as:
Step 1: [Your first reasoning step]
Step 2: [Your next reasoning step]
...
Final Answer: [Your conclusive answer]



In [10]:
messages = [
    SystemMessage(content=(
        "You are a clear, careful assistant. Do NOT reveal private chain-of-thought. "
        "Provide a concise final answer and a brief, high-level, non-sensitive stepwise summary "
        "explaining the approach (3–8 numbered steps). Keep the internal detailed reasoning private."
    )),
    HumanMessage(content=prompt),
]

# Call the model
response = llm.invoke(messages)  # returns a ChatMessage object
result_text = response.content

# Print result
print("\n--- MODEL OUTPUT ---\n")
print(result_text)


--- MODEL OUTPUT ---

Step 1: Identify that the problem is asking for the total number of apples the store has after selling 8 and receiving 12 more.
Step 2: Note that the store starts with 15 apples, sells 8, and then receives 12 more.
Step 3: Calculate the total number of apples by adding the initial apples, subtracting the sold apples, and adding the received apples.
Step 4: Set up the calculation: 15 (initial) - 8 (sold) + 12 (received).
Step 5: Perform the calculation: 15 - 8 + 12 = 19.
Step 6: Verify that the answer makes sense in the context of the problem.

Final Answer: The store now has 19 apples.


## **Alternative Prompt**

In [11]:
complex_prompt = f"""
You are an expert problem solver. Use chain of thought reasoning to break down this problem into manageable steps.

Problem: {selected_problem}

Think through this carefully:

First, let me understand what's being asked:
[Your initial analysis]

Now, let me identify the key information:
[Extract relevant facts and data]

Next, I'll determine the approach:
[Outline your solution strategy]

Then, I'll work through the steps:
[Detailed step-by-step reasoning]

Finally, I'll verify the solution:
[Check your work and confirm]

Answer:
"""

In [12]:
messages = [
    SystemMessage(content=(
        "You are a clear, careful assistant. Do NOT reveal private chain-of-thought. "
        "Provide a concise final answer and a brief, high-level, non-sensitive stepwise summary "
        "explaining the approach (3–8 numbered steps). Keep the internal detailed reasoning private."
    )),
    HumanMessage(content=complex_prompt),
]

# Call the model
complex_response = llm.invoke(messages)  # returns a ChatMessage object
result_text = complex_response.content

# Print result
print("\n--- MODEL OUTPUT ---\n")
print(result_text)


--- MODEL OUTPUT ---

Answer: The store now has 19 apples.

1. Initial analysis: We need to find the total number of apples the store has after selling 8 and receiving a shipment of 12 apples.
2. Relevant facts: The store starts with 15 apples, sells 8, and receives 12 more apples.
3. Solution strategy: Calculate the total number of apples by starting with the initial quantity, subtracting the sold apples, and then adding the received apples.
4. Calculate: Start with 15 apples, subtract 8 sold apples to get 7 apples, then add 12 received apples to get a total of 19 apples.
5. Verify: 15 (initial) - 8 (sold) + 12 (received) = 19 apples, confirming the solution.
